# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset package using the [mlcroissant](https://github.com/mlcommons/croissant) library, referencing all entities by their `@id` as specified in the Croissant schema. It covers data loading, overview, extraction, EDA, and visualization.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Install mlcroissant if not already available
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step will fetch the Croissant schema and prepare the dataset for structured access via `@id` references.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a DatasetMetadata object

print(f"Dataset title: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Version: {getattr(metadata, 'version', None)}\n")

## 2. Data Overview
Let's review the available record sets, their `@id`s, as well as the fields and their `@id`s, to understand the structure of the FAIR² dataset.

We will list all record sets, displaying their `@id` and fields for reference.

In [ ]:
# List record sets and field @id's
record_set_infos = []
for rs in metadata.record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    fields = getattr(rs, 'fields', [])
    print(f"RecordSet: {rs_name if rs_name else ''} (@id: {rs_id})")
    for field in fields:
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f"  Field: {field_name if field_name else ''} (@id: {field_id})")
    print()

## 3. Data Extraction
We will load the records from each available record set into a pandas DataFrame for analysis, identifying both record sets and fields by their `@id` as recommended.

We first collect all record set `@id`s.

In [ ]:
# Gather all record set @ids
record_set_ids = [getattr(rs, '@id', None) for rs in metadata.record_sets]
print("Record set @ids:")
for rs_id in record_set_ids:
    print("  ", rs_id)

# Load each record set to a DataFrame by @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {df.shape[0]} records for record set @id: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Show columns for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in first record set (@id={first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We select a numeric field by `@id` from the chosen record set (`@id` shown above), filter and normalize its values, and group by another attribute if available.

> **Note:** Replace the `<numeric_field_id>` and `<group_field_id>` with valid `@id`s you found above, or run the cell below to print all columns for hints.

In [ ]:
# -- Example selection, user should adapt field @ids as needed -- #
# Suppose main patient record set has an @id such as 'cr:Patient', adapt if different
main_rs_id = record_set_ids[0] if record_set_ids else None
# Print all columns to assist manual selection
if main_rs_id is not None:
    print("Columns available in the main record set:")
    print(dataframes[main_rs_id].columns.tolist())

# Example (replace with correct @id from previous cell!):
# numeric_field_id = '@id-of-numeric-field'   e.g., 'cr:Age' if present
# group_field_id = '@id-of-grouping-field'   e.g., 'cr:Sex' or 'cr:MSI_status'

# For demonstration, auto-select first numeric column & a possible group field
import numpy as np
df = dataframes.get(main_rs_id)
numeric_field_id, group_field_id = None, None
if df is not None:
    for c in df.columns:
        if np.issubdtype(df[c].dropna().apply(type).mode()[0], np.number):
            numeric_field_id = c
            break
    # Use the first non-numeric as group field
    for c in df.columns:
        if c != numeric_field_id:
            group_field_id = c
            break
    print(f"Auto-selected numeric field for analysis: {numeric_field_id}")
    print(f"Auto-selected group field: {group_field_id}")
else:
    print("No main record set loaded.")

# Proceed if both exist
if df is not None and numeric_field_id is not None:
    # Convert to numeric, if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Example threshold: use median
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std()!=0 else 1)

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field and show mean
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
We'll produce a simple visualization, such as a histogram of the selected numeric field and a boxplot grouped by the group field if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, reference, and analyze the FAIR² dataset by record set and field `@id` as specified in the dataset's Croissant schema.

- All data access referenced entities using their `@id` fields for full reproducibility.
- We extracted tabular data from each record set, normalized numeric fields, and grouped data for basic exploratory analysis.
- Visualizations helped summarize the data distributions and relationships.

For further analysis, refer to the Croissant schema for more precise variable annotations, and adapt the code to target more complex data processing or modelling workflows.